# Notebook 3 — Testing & Experiments
**ISOM5240 Group Project — Earnings Call Dividend Risk Early Warning System**

This notebook runs the full 3-pipeline Danger Index system on test transcripts
and generates a comparative experiment table.

**Experiments:**
1. Model selection: Base vs Fine-tuned (accuracy + runtime)
2. Pipeline ablation: individual pipelines vs full system
3. Full system accuracy on known dividend-event transcripts
4. Application performance on Streamlit Cloud
5. Export results to Excel


In [7]:
# ── Authenticate with HuggingFace Hub via Colab Secrets ─────────────────────
import os
from google.colab import userdata
from huggingface_hub import login

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    login(token=HF_TOKEN, add_to_git_credential=True)
    print('✅ Logged in to HuggingFace Hub successfully')
except Exception as e:
    print(f'⚠️  Could not log in automatically: {e}')
    print('   Falling back to manual login...')
    from huggingface_hub import notebook_login
    notebook_login()

⚠️  Could not log in automatically: Secret HF_TOKEN does not exist.
   Falling back to manual login...


In [1]:
# ── Step 1: Install packages ─────────────────────────────────────────────────
!pip install -q transformers datasets accelerate scikit-learn huggingface_hub openpyxl

In [5]:
# ── Step 2: Imports ─────────────────────────────────────────────────────────
import time, re
import numpy as np
import pandas as pd
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import torch
print('GPU available:', torch.cuda.is_available())

GPU available: False


In [4]:
# ── Step 3: Configure model IDs ─────────────────────────────────────────────
# ⚠️  Update these to your actual HuggingFace Hub IDs after pushing from Notebooks 1 & 2

HF_USERNAME        = 'ruirui0506'
SENTIMENT_MODEL_ID = f'{HF_USERNAME}/finbert-dividend-sentiment'
RISK_MODEL_ID      = f'{HF_USERNAME}/dividend-risk-bert'
ZEROSHOT_MODEL_ID  = 'facebook/bart-large-mnli'

DANGER_LABELS = [
    'dividend cut risk', 'liquidity stress',
    'debt covenant breach', 'capital preservation', 'payout suspension',
]

In [8]:
# ── Step 4: Load all three models ───────────────────────────────────────────
print('Loading Pipeline 1 — FinBERT sentiment ...')
sent_tok  = AutoTokenizer.from_pretrained(SENTIMENT_MODEL_ID)
sent_mdl  = AutoModelForSequenceClassification.from_pretrained(SENTIMENT_MODEL_ID)
sent_pipe = pipeline('text-classification', model=sent_mdl,
                     tokenizer=sent_tok, device=-1, top_k=None)

print('Loading Pipeline 2 — Dividend Risk classifier ...')
risk_tok  = AutoTokenizer.from_pretrained(RISK_MODEL_ID)
risk_mdl  = AutoModelForSequenceClassification.from_pretrained(RISK_MODEL_ID)
risk_pipe = pipeline('text-classification', model=risk_mdl,
                     tokenizer=risk_tok, device=-1, top_k=None)

print('Loading Pipeline 3 — BART zero-shot ...')
zs_pipe = pipeline('zero-shot-classification',
                   model=ZEROSHOT_MODEL_ID, device=-1)

print('\n✅ All models loaded.')

Loading Pipeline 1 — FinBERT sentiment ...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading Pipeline 2 — Dividend Risk classifier ...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading Pipeline 3 — BART zero-shot ...


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]


✅ All models loaded.


## PIPELINE 1 — Sentiment Model Selection (Fine-tuned)
**Template Step 1 → Step 2a → Step 2b**

Compare THREE pre-trained sentiment models by fine-tuning each on the
Financial PhraseBank dataset and recording accuracy + runtime.
The best model is then used as Pipeline 1 in the deployed app.

Candidate models:
- **Model 1:** `ProsusAI/finbert` — standard FinBERT baseline
- **Model 2:** `yiyanghkust/finbert-tone` — FinBERT trained on earnings forward-looking statements
- **Model 3:** `ahmedrachid/FinancialBERT-Sentiment-Analysis` — FinancialBERT fine-tuned on FiQA


In [9]:
# ── Pipeline 1 Step A: Load Financial PhraseBank and prepare splits ─────────
# This is the same dataset used in Notebook 1.
# We use a lightweight 3-epoch training here purely for model SELECTION.
# The winning model is then fully fine-tuned in Notebook 1.

import os, time, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback,
)
import torch

# ── Download Financial PhraseBank ────────────────────────────────────────────
!wget -q https://github.com/neoyipeng2018/FinancialPhraseBank-v1.0/raw/main/FinancialPhraseBank-v1.0.zip \
    -O FinancialPhraseBank-v1.0.zip
import zipfile
with zipfile.ZipFile('FinancialPhraseBank-v1.0.zip','r') as z:
    z.extractall()

df_pb = pd.read_csv(
    'FinancialPhraseBank-v1.0/Sentences_AllAgree.txt',
    sep='@', header=None, names=['sentence','label'], encoding='iso-8859-1'
)
LABEL2ID_P1 = {'negative':0,'neutral':1,'positive':2}
ID2LABEL_P1 = {v:k for k,v in LABEL2ID_P1.items()}
df_pb['label'] = df_pb['label'].str.strip().map(LABEL2ID_P1)
assert df_pb['label'].isna().sum() == 0

train_pb, temp_pb = train_test_split(df_pb, test_size=0.2, stratify=df_pb['label'], random_state=42)
val_pb,  test_pb  = train_test_split(temp_pb, test_size=0.5, stratify=temp_pb['label'], random_state=42)

p1_dataset = DatasetDict({
    'train':      Dataset.from_pandas(train_pb.rename(columns={'label':'labels'}), preserve_index=False),
    'validation': Dataset.from_pandas(val_pb.rename(columns={'label':'labels'}),   preserve_index=False),
    'test':       Dataset.from_pandas(test_pb.rename(columns={'label':'labels'}),  preserve_index=False),
})
print(f'P1 dataset — Train: {len(train_pb)}, Val: {len(val_pb)}, Test: {len(test_pb)}')
print('Class distribution (train):')
print(train_pb['label'].value_counts().sort_index().rename(ID2LABEL_P1))


P1 dataset — Train: 1811, Val: 226, Test: 227
Class distribution (train):
label
negative     242
neutral     1113
positive     456
Name: count, dtype: int64


In [11]:
# ── Pipeline 1 Step B: Fine-tune 3 candidate models and compare ─────────────

P1_CANDIDATES = [
    ('Model 1', 'ProsusAI/finbert'),
    ('Model 2', 'yiyanghkust/finbert-tone'),
    ('Model 3', 'ahmedrachid/FinancialBERT-Sentiment-Analysis'),
]

def compute_metrics_p1(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1':       f1_score(labels, preds, average='weighted'),
    }

p1_results = []

for model_label, model_id in P1_CANDIDATES:
    print(f'\n{'='*60}')
    print(f'Fine-tuning {model_label}: {model_id}')
    print('='*60)

    # Tokenize with this model's own tokenizer
    tok = AutoTokenizer.from_pretrained(model_id)
    def tokenize_fn(examples):
        return tok(examples['sentence'], truncation=True,
                   padding='max_length', max_length=128)
    tokenized_p1 = p1_dataset.map(tokenize_fn, batched=True)
    tokenized_p1 = tokenized_p1.rename_column('label', 'labels') \
        if 'label' in tokenized_p1['train'].column_names else tokenized_p1
    tokenized_p1.set_format('torch', columns=['input_ids','attention_mask','labels'])

    # Load model
    mdl = AutoModelForSequenceClassification.from_pretrained(
        model_id, num_labels=3,
        id2label=ID2LABEL_P1, label2id=LABEL2ID_P1,
        ignore_mismatched_sizes=True,
    )

    # Training args — 3 epochs only, for model selection speed
    args = TrainingArguments(
        output_dir              = f'./p1_selection_{model_label.replace(" ","_")}',
        num_train_epochs        = 3,
        per_device_train_batch_size = 16,
        per_device_eval_batch_size  = 32,
        learning_rate           = 2e-5,
        weight_decay            = 0.01,
        warmup_ratio            = 0.1,
        eval_strategy           = 'epoch',
        save_strategy           = 'epoch',
        load_best_model_at_end  = True,
        metric_for_best_model   = 'f1',
        greater_is_better       = True,
        logging_steps           = 50,
        fp16                    = torch.cuda.is_available(),
        report_to               = 'none',
    )

    trainer = Trainer(
        model            = mdl,
        args             = args,
        train_dataset    = tokenized_p1['train'],
        eval_dataset     = tokenized_p1['validation'],
        processing_class = tok,
        compute_metrics  = compute_metrics_p1,
        callbacks        = [EarlyStoppingCallback(early_stopping_patience=2)],
    )

    t0 = time.time()
    trainer.train()
    runtime = round(time.time() - t0, 1)

    # Evaluate on test set
    test_out  = trainer.predict(tokenized_p1['test'])
    y_pred    = np.argmax(test_out.predictions, axis=-1)
    y_true    = test_out.label_ids
    test_acc  = round(accuracy_score(y_true, y_pred), 3)

    p1_results.append({
        'Model':    model_label,
        'Model Name': model_id,
        'Accuracy (after fine-tuning)': test_acc,
        'Run time (in sec.)': runtime,
    })
    print(f'  → Accuracy: {test_acc:.3f} | Runtime: {runtime}s')

    # Clean up to free GPU memory
    del mdl, trainer
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

p1_selection_df = pd.DataFrame(p1_results)
print('\n── Pipeline 1 Model Selection Results ──────────────────────')
display(p1_selection_df)



Fine-tuning Model 1: ProsusAI/finbert


Map:   0%|          | 0/1811 [00:00<?, ? examples/s]

Map:   0%|          | 0/226 [00:00<?, ? examples/s]

Map:   0%|          | 0/227 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
# ── Pipeline 1 Step C: Select best model ────────────────────────────────────

# Best model = highest accuracy; break ties by fastest runtime
best_p1_row = p1_selection_df.sort_values(
    ['Accuracy (after fine-tuning)', 'Run time (in sec.)'],
    ascending=[False, True]
).iloc[0]

best_p1_label = best_p1_row['Model']
best_p1_id    = best_p1_row['Model Name']
best_p1_acc   = best_p1_row['Accuracy (after fine-tuning)']
best_p1_rt    = best_p1_row['Run time (in sec.)']

print('── Pipeline 1 Model Selection Decision ─────────────────────')
print(f'  Selected : {best_p1_label} — {best_p1_id}')
print(f'  Accuracy : {best_p1_acc:.3f}')
print(f'  Runtime  : {best_p1_rt}s')
print(f'\n  Reason   : Highest fine-tuned accuracy on Financial PhraseBank test set.')
print(f'  Action   : {best_p1_id} is fully fine-tuned in Notebook 1')
print(f'             and uploaded to HuggingFace Hub as the Pipeline 1 model.')


---
## PIPELINE 2 — Dividend Risk Classifier Model Selection (Fine-tuned)
**Template Step 1 → Step 2a → Step 2b**

Compare THREE pre-trained BERT-family models by fine-tuning each on the
custom dividend risk dataset and recording accuracy + runtime.
The best model is used as Pipeline 2 in the deployed app.

Candidate models:
- **Model 1:** `distilbert-base-uncased` — lightweight, fast, good for deployment
- **Model 2:** `bert-base-uncased` — full BERT, larger and more powerful
- **Model 3:** `albert-base-v2` — parameter-efficient, faster inference than BERT


In [ ]:
# ── Pipeline 2 Step A: Build the dividend risk dataset ───────────────────────
# Replicates the dataset construction from Notebook 2.
# Split FIRST on seed data, then augment training split only (no data leakage).

seed_data = [
    # LOW RISK (0)
    ('Our dividend has grown for 25 consecutive years and we remain committed to continued growth reflecting our strong free cash flow.', 0),
    ('The board approved a 6% increase in the quarterly dividend given our confidence in earnings sustainability and balance sheet strength.', 0),
    ('Free cash flow reached a record 3.2 billion dollars providing 2.8 times coverage of our total dividend commitments.', 0),
    ('Net debt decreased by 1.2 billion year-to-date and we remain on track to reach our leverage target of 2.0 times by year end.', 0),
    ('Our payout ratio of 42% is well within our target range and provides significant headroom even in a stress scenario.', 0),
    ('Liquidity stands at 4.5 billion including undrawn credit facilities providing ample financial flexibility for dividends and buybacks.', 0),
    ('Capital returns to shareholders remain our top priority; we repurchased 500 million in shares and paid 800 million in dividends this quarter.', 0),
    ('We expect to generate 2.5 to 2.8 billion in free cash flow for the full year fully supporting our dividend and repurchase program.', 0),
    ('The board reaffirmed its commitment to the dividend as a cornerstone of our capital allocation strategy.', 0),
    ('Our interest coverage ratio stands at a healthy 8.2 times well above our internal threshold and covenant requirements.', 0),
    ('We have no material debt maturities until 2027 and our refinancing risk is minimal.', 0),
    ('Adjusted EBITDA increased 18% year-over-year demonstrating strong operating leverage and execution against our strategic plan.', 0),
    ('Our investment-grade credit rating was reaffirmed this quarter reflecting our conservative financial management and strong cash generation.', 0),
    ('We are confident in our ability to sustain and grow the dividend over the long term underpinned by our business model resilience.', 0),
    ('Dividend coverage from operating cash flow improved to 3.1 times compared to 2.8 times in the prior year period.', 0),
    ('Return on invested capital improved to 14.5% demonstrating excellent capital discipline and value creation for shareholders.', 0),
    ('We increased the quarterly cash dividend by 8 cents per share representing a 7% increase over the prior period.', 0),
    ('Debt-to-EBITDA declined to 1.8 times reflecting continued deleveraging and our commitment to maintaining a strong balance sheet.', 0),
    ('Our strong operational performance translated into exceptional cash generation enabling us to fund all capital priorities comfortably.', 0),
    ('The strength of our diversified revenue streams gives us confidence to raise full-year guidance and affirm our dividend trajectory.', 0),
    ('Share repurchases of 300 million were completed this quarter and we have 1.2 billion remaining under our current buyback authorization.', 0),
    ('Revenue grew 15% year-over-year driven by strong volume and favorable pricing dynamics across all our business segments.', 0),
    ('Our financial position is the strongest it has been in a decade allowing us to both invest in growth and return capital to shareholders.', 0),
    ('We are extremely pleased with the performance of our core business segments which continue to generate substantial and growing free cash flow.', 0),
    ('We are executing well on our cost reduction program and expect to achieve 400 million in savings ahead of schedule.', 0),
    # MEDIUM RISK (1)
    ('While we continue to support our dividend we are monitoring market conditions carefully and will reassess our capital allocation priorities if needed.', 1),
    ('Our payout ratio has risen to 72% this quarter which is elevated relative to historical levels; we are focused on improving free cash flow.', 1),
    ('Management is conducting a comprehensive review of our capital allocation framework including the appropriate level of the dividend going forward.', 1),
    ('We expect free cash flow to be temporarily impacted by elevated capital expenditures this year before recovering to normalized levels in the following year.', 1),
    ('The macroeconomic environment remains uncertain and we are managing our cost base and capital spending prudently to preserve financial flexibility.', 1),
    ('Revenue declined 8% year-over-year due to softening demand in our key end markets and pricing pressure from competitive dynamics.', 1),
    ('We drew down approximately 300 million on our revolving credit facility in the quarter to fund working capital needs during a period of operational transition.', 1),
    ('Our leverage ratio increased to 3.5 times this quarter due to the recent acquisition; we expect to deleverage to below 3.0 times within 18 months.', 1),
    ('While our dividend remains a priority we acknowledge the payout ratio is higher than we would like and we are taking steps to reduce it through earnings growth.', 1),
    ('Management continues to evaluate the appropriate balance between debt reduction, capital investment, and cash returns to shareholders.', 1),
    ('Our outlook for the remainder of the year is cautious given macroeconomic headwinds though we expect to maintain our current dividend level.', 1),
    ('The board discussed the dividend policy at its most recent meeting and determined to maintain the current level while monitoring performance.', 1),
    ('Certain covenant thresholds require us to maintain leverage below 4.0 times which limits our financial flexibility in the near term.', 1),
    ('Cash flow from operations decreased 22% year-over-year primarily due to higher working capital requirements and one-time tax payments.', 1),
    ('We have initiated a strategic review process that will include an assessment of our capital structure and dividend policy.', 1),
    ('We are in active dialogue with our lenders regarding an amendment to certain financial covenants to provide additional headroom.', 1),
    ('We acknowledge the current payout ratio is above our long-term target and we are committed to growing earnings to bring it back in line.', 1),
    ('While no decision has been made we are evaluating all options for optimizing our capital structure including the dividend level.', 1),
    ('Operating margins contracted 300 basis points due to higher input costs which we expect to partially offset through pricing actions.', 1),
    ('Our liquidity position has decreased to 1.2 billion which while adequate requires us to be thoughtful about capital allocation decisions.', 1),
    ('While we remain committed to returning capital to shareholders we may moderate the pace of share repurchases to prioritize balance sheet strength.', 1),
    ('Our near-term earnings visibility is reduced by elevated input cost inflation and we are working to pass through price increases.', 1),
    ('The integration of our recent acquisition is ongoing and until completion our free cash flow may be temporarily impacted by one-time costs.', 1),
    ('We are proactively managing our cost structure in response to volume declines and expect to return to positive free cash flow in the second half.', 1),
    ('We are experiencing timing differences in working capital that have temporarily reduced our reported free cash flow; underlying cash generation remains solid.', 1),
    # HIGH RISK (2)
    ('Free cash flow turned negative this quarter and we are actively reviewing all components of our capital allocation including the sustainability of the dividend.', 2),
    ('Our payout ratio has risen to 127% of free cash flow which is clearly unsustainable and the board is evaluating corrective actions.', 2),
    ('We are prioritizing capital preservation and debt reduction; the board is reviewing whether the current dividend level can be maintained.', 2),
    ('Liquidity has decreased to 680 million and we have drawn substantially on our revolving credit facility leaving limited available capacity.', 2),
    ('Our net leverage ratio has increased to 5.8 times approaching our covenant threshold of 6.0 times which would trigger a default event.', 2),
    ('Revenue declined 28% year-over-year and we are implementing emergency cost reduction measures to protect the viability of our operations.', 2),
    ('The board will convene a special meeting to discuss strategic alternatives including a potential reduction or suspension of the quarterly dividend.', 2),
    ('We have engaged financial advisors to assist in evaluating our capital structure options given the deterioration in our financial performance.', 2),
    ('Our interest coverage ratio has declined to 1.4 times which raises significant concerns about our ability to service our debt obligations.', 2),
    ('We received a notice from our lenders regarding a covenant breach and are currently in discussions to obtain a waiver or amendment.', 2),
    ('Cash and equivalents have fallen to 220 million and we must take immediate action to address our liquidity position.', 2),
    ('The combination of declining earnings, elevated capex requirements, and rising interest expense has created an untenable cash flow deficit.', 2),
    ('Management is exploring all strategic options including asset divestitures, equity issuance, and a restructuring of our capital return program.', 2),
    ('We suspended our share repurchase program and are conducting a thorough review of the dividend in light of current financial pressures.', 2),
    ('Our ability to continue paying the dividend at the current level is dependent on a successful refinancing of our near-term debt maturities.', 2),
    ('Given the severity of the earnings decline and the need to preserve capital the board has decided to reduce the quarterly dividend by 50%.', 2),
    ('We have failed to comply with certain financial covenants and our lenders have accelerated 800 million of outstanding debt.', 2),
    ('The dividend has been suspended indefinitely to preserve cash as the company undergoes a comprehensive financial restructuring.', 2),
    ('Free cash flow was negative 340 million this quarter and we project continued negative free cash flow in the first half of next year.', 2),
    ('We are in discussions with a financial restructuring advisor and have retained legal counsel to evaluate all available options.', 2),
    ('Our business has experienced an unprecedented deterioration in demand and cash flow requiring immediate and decisive capital allocation changes.', 2),
    ('The board approved a 75% reduction in the quarterly dividend to prioritize debt repayment and financial stability.', 2),
    ('We will not be in a position to maintain the current dividend if operating conditions do not improve materially in the next two quarters.', 2),
    ('Total debt stands at 8.2 billion against EBITDA of 900 million; this leverage level is not sustainable and requires immediate remediation.', 2),
    ('We have insufficient cash generation to fund both our capital expenditure requirements and the current dividend simultaneously.', 2),
]

import random
random.seed(42)

FINANCIAL_SYNONYMS = {
    'free cash flow':  ['operating cash flow','cash generation','cash from operations'],
    'dividend':        ['distribution','quarterly payment','shareholder payout'],
    'payout ratio':    ['dividend coverage ratio','distribution ratio'],
    'balance sheet':   ['financial position','capital structure'],
    'liquidity':       ['financial flexibility','cash position'],
    'leverage':        ['debt levels','indebtedness'],
    'revenue':         ['sales','top-line','net revenues'],
    'shareholders':    ['investors','stockholders'],
}

def augment_text(text):
    a = text.lower()
    for orig, alts in FINANCIAL_SYNONYMS.items():
        if orig in a and random.random() < 0.6:
            a = a.replace(orig, random.choice(alts), 1)
    return a[0].upper() + a[1:]

df_seed = pd.DataFrame(seed_data, columns=['text','label'])

# ── Split FIRST (no leakage), then augment training only ─────────────────────
train_seed, temp_s = train_test_split(df_seed, test_size=0.25,
                                      stratify=df_seed['label'], random_state=42)
val_s, test_s = train_test_split(temp_s, test_size=0.5,
                                 stratify=temp_s['label'], random_state=42)

aug_rows = []
for _, row in train_seed.iterrows():
    for _ in range(2):
        aug_rows.append({'text': augment_text(row['text']), 'label': row['label']})
train_s = pd.concat([train_seed, pd.DataFrame(aug_rows)], ignore_index=True
                    ).sample(frac=1, random_state=42).reset_index(drop=True)

p2_dataset = DatasetDict({
    'train':      Dataset.from_pandas(train_s.rename(columns={'label':'labels'}), preserve_index=False),
    'validation': Dataset.from_pandas(val_s.rename(columns={'label':'labels'}),   preserve_index=False),
    'test':       Dataset.from_pandas(test_s.rename(columns={'label':'labels'}),  preserve_index=False),
})
print(f'P2 dataset — Train: {len(train_s)}, Val: {len(val_s)}, Test: {len(test_s)}')
print('Class distribution (train):')
print(train_s['label'].value_counts().sort_index().rename({0:'Low',1:'Medium',2:'High'}))


In [ ]:
# ── Pipeline 2 Step B: Fine-tune 3 candidate models and compare ─────────────
import torch.nn as nn
from sklearn.utils.class_weight import compute_class_weight

P2_CANDIDATES = [
    ('Model 1', 'distilbert-base-uncased'),
    ('Model 2', 'bert-base-uncased'),
    ('Model 3', 'albert-base-v2'),
]

ID2LABEL_P2 = {0:'Low Risk', 1:'Medium Risk', 2:'High Risk'}
LABEL2ID_P2 = {v:k for k,v in ID2LABEL_P2.items()}

# Class weights computed once from training split
cw = compute_class_weight('balanced', classes=np.array([0,1,2]),
                           y=train_s['label'].values)
class_weights_p2 = torch.tensor(cw, dtype=torch.float)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop('labels')
        outputs = model(**inputs)
        loss_fn = nn.CrossEntropyLoss(
            weight=class_weights_p2.to(outputs.logits.device)
        )
        loss = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics_p2(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy':   accuracy_score(labels, preds),
        'f1_macro':   f1_score(labels, preds, average='macro'),
    }

p2_results = []

for model_label, model_id in P2_CANDIDATES:
    print(f'\n{"="*60}')
    print(f'Fine-tuning {model_label}: {model_id}')
    print('='*60)

    tok = AutoTokenizer.from_pretrained(model_id)
    def tokenize_fn_p2(examples):
        return tok(examples['text'], truncation=True,
                   padding='max_length', max_length=128)

    tokenized_p2 = p2_dataset.map(tokenize_fn_p2, batched=True)
    tokenized_p2.set_format('torch', columns=['input_ids','attention_mask','labels'])

    mdl = AutoModelForSequenceClassification.from_pretrained(
        model_id, num_labels=3,
        id2label=ID2LABEL_P2, label2id=LABEL2ID_P2,
        ignore_mismatched_sizes=True,
    )

    args = TrainingArguments(
        output_dir              = f'./p2_selection_{model_label.replace(" ","_")}',
        num_train_epochs        = 6,
        per_device_train_batch_size = 16,
        per_device_eval_batch_size  = 32,
        learning_rate           = 3e-5,
        weight_decay            = 0.01,
        warmup_ratio            = 0.15,
        eval_strategy           = 'epoch',
        save_strategy           = 'epoch',
        load_best_model_at_end  = True,
        metric_for_best_model   = 'f1_macro',
        greater_is_better       = True,
        logging_steps           = 10,
        fp16                    = torch.cuda.is_available(),
        report_to               = 'none',
    )

    trainer = WeightedTrainer(
        model            = mdl,
        args             = args,
        train_dataset    = tokenized_p2['train'],
        eval_dataset     = tokenized_p2['validation'],
        processing_class = tok,
        compute_metrics  = compute_metrics_p2,
        callbacks        = [EarlyStoppingCallback(early_stopping_patience=3)],
    )

    t0 = time.time()
    trainer.train()
    runtime = round(time.time() - t0, 1)

    test_out = trainer.predict(tokenized_p2['test'])
    y_pred   = np.argmax(test_out.predictions, axis=-1)
    y_true   = test_out.label_ids
    test_acc = round(accuracy_score(y_true, y_pred), 3)
    test_f1  = round(f1_score(y_true, y_pred, average='macro'), 3)

    p2_results.append({
        'Model':      model_label,
        'Model Name': model_id,
        'Accuracy (after fine-tuning)': test_acc,
        'Macro F1':   test_f1,
        'Run time (in sec.)': runtime,
    })
    print(f'  → Accuracy: {test_acc:.3f} | Macro F1: {test_f1:.3f} | Runtime: {runtime}s')

    del mdl, trainer
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

p2_selection_df = pd.DataFrame(p2_results)
print('\n── Pipeline 2 Model Selection Results ──────────────────────')
display(p2_selection_df)


In [ ]:
# ── Pipeline 2 Step C: Select best model ────────────────────────────────────

# Best model = highest macro F1 (most balanced across all 3 risk classes)
# Break ties by fastest runtime (important for Streamlit Cloud deployment)
best_p2_row = p2_selection_df.sort_values(
    ['Macro F1', 'Run time (in sec.)'],
    ascending=[False, True]
).iloc[0]

best_p2_label = best_p2_row['Model']
best_p2_id    = best_p2_row['Model Name']
best_p2_acc   = best_p2_row['Accuracy (after fine-tuning)']
best_p2_f1    = best_p2_row['Macro F1']
best_p2_rt    = best_p2_row['Run time (in sec.)']

print('── Pipeline 2 Model Selection Decision ─────────────────────')
print(f'  Selected : {best_p2_label} — {best_p2_id}')
print(f'  Accuracy : {best_p2_acc:.3f}')
print(f'  Macro F1 : {best_p2_f1:.3f}')
print(f'  Runtime  : {best_p2_rt}s')
print(f'\n  Reason   : Highest macro F1 across all three risk classes,')
print(f'             ensuring balanced detection of Low / Medium / High risk.')
print(f'  Action   : {best_p2_id} is fully fine-tuned in Notebook 2')
print(f'             and uploaded to HuggingFace Hub as the Pipeline 2 model.')


---
## APP EVALUATION — 10-Sample Test on Deployed Streamlit App
**Template Step 3: Implement the App and evaluate overall performance**

Run the full 3-pipeline system on 10 labeled transcript samples.
Record: Input (transcript excerpt) | Output (predicted signal + index) | Correct or not
Final accuracy reported as X out of 10.


In [ ]:
# ── App Evaluation Step A: Define 10 labeled test samples ────────────────────
# Ground truth: 0 = dividend safe (expect HEALTHY or CAUTION)
#               1 = dividend cut/suspended (expect DANGER or CRITICAL)
# These are DIFFERENT from the 6 samples used in Step 6 to avoid overlap.

APP_TEST_CASES = [
    {
        'id': 'Test sample 1',
        'ground_truth': 0,
        'transcript': 'We delivered record earnings this quarter with EPS up 19% '
            'year-over-year. Free cash flow of 1.6 billion provided 2.5 times '
            'coverage of our dividend. The board declared a 9% dividend increase '
            'effective next quarter, reflecting confidence in our long-term earnings '
            'power. Net debt declined to 1.4 times EBITDA. Our liquidity position '
            'remains robust at 3.8 billion with no near-term debt maturities.',
    },
    {
        'id': 'Test sample 2',
        'ground_truth': 0,
        'transcript': 'Revenue grew modestly at 4% while input cost inflation '
            'compressed margins by 120 basis points. Free cash flow was adequate '
            'but below prior year levels. Our payout ratio rose to 61%. We remain '
            'committed to the dividend and expect margin recovery in the second half '
            'as pricing actions take effect. No changes to our capital return policy.',
    },
    {
        'id': 'Test sample 3',
        'ground_truth': 1,
        'transcript': 'This has been an extraordinarily challenging quarter. '
            'Revenue fell 32% and free cash flow was deeply negative at minus 410 million. '
            'We have drawn our entire revolving credit facility and liquidity stands '
            'at 290 million. The board has suspended the quarterly dividend indefinitely '
            'to preserve capital. We are in active discussions with lenders about a '
            'comprehensive balance sheet restructuring.',
    },
    {
        'id': 'Test sample 4',
        'ground_truth': 0,
        'transcript': 'Our infrastructure segment delivered 11% organic revenue growth '
            'supported by long-term contracted cash flows. Distributable cash flow '
            'covered the dividend 1.9 times. We raised the quarterly distribution by '
            '5 cents per unit representing a 6% annualized increase. Leverage of '
            '3.2 times is within our target range and we expect further reduction '
            'through organic growth over the next 12 months.',
    },
    {
        'id': 'Test sample 5',
        'ground_truth': 1,
        'transcript': 'I want to address the elephant in the room regarding our dividend. '
            'Cash generation has been severely impaired by the structural decline in '
            'our legacy business. The payout ratio on free cash flow basis reached '
            '148% this quarter. After careful deliberation the board has approved a '
            '65% reduction in the quarterly dividend to redirect capital toward debt '
            'repayment and reinvestment in growth initiatives.',
    },
    {
        'id': 'Test sample 6',
        'ground_truth': 0,
        'transcript': 'We are navigating a period of transition following our major '
            'acquisition. Integration is progressing ahead of schedule. Leverage '
            'increased to 3.8 times but we are on track to reduce it to below 3.0 '
            'times within 24 months. Free cash flow was temporarily impacted by '
            'integration costs but underlying generation is solid. We are maintaining '
            'the dividend at current levels throughout the integration period.',
    },
    {
        'id': 'Test sample 7',
        'ground_truth': 1,
        'transcript': 'Our leverage has now reached 6.1 times EBITDA breaching our '
            'maintenance covenant of 6.0 times. We have received a formal default '
            'notice from our lending syndicate and are seeking a waiver. Cash on hand '
            'is 145 million which is insufficient to service upcoming debt maturities. '
            'Under these circumstances the board has determined it cannot continue '
            'paying the dividend and has voted to eliminate it effective immediately.',
    },
    {
        'id': 'Test sample 8',
        'ground_truth': 0,
        'transcript': 'We generated 2.1 billion in operating cash flow this quarter, '
            'a 14% increase year-over-year. Capital expenditures were 380 million, '
            'leaving free cash flow of 1.72 billion. Dividend payments totaled '
            '620 million representing a comfortable payout ratio of 36%. We are '
            'on track to return over 3 billion to shareholders through dividends '
            'and buybacks for the full fiscal year.',
    },
    {
        'id': 'Test sample 9',
        'ground_truth': 1,
        'transcript': 'Volumes in our core division declined 41% due to a catastrophic '
            'loss of a major customer contract. We are burning approximately '
            '80 million in cash per month at current run rates. Management has '
            'engaged a restructuring advisor and is evaluating all alternatives '
            'including asset sales, equity issuance, and a potential suspension '
            'of the dividend to stabilize the business.',
    },
    {
        'id': 'Test sample 10',
        'ground_truth': 0,
        'transcript': 'Our REIT portfolio generated same-store NOI growth of 5.2% '
            'supported by strong occupancy of 97.4% and embedded rent escalations. '
            'Adjusted funds from operations of 1.42 per share covered the quarterly '
            'distribution of 0.78 per share by 1.82 times. We increased our full-year '
            'AFFO guidance and reaffirm our commitment to growing the distribution '
            'in line with earnings per share growth.',
    },
]
print(f'{len(APP_TEST_CASES)} test samples defined.')
print(f'Safe (ground truth 0): {sum(1 for x in APP_TEST_CASES if x["ground_truth"]==0)}')
print(f'Cut  (ground truth 1): {sum(1 for x in APP_TEST_CASES if x["ground_truth"]==1)}')


In [ ]:
# ── App Evaluation Step B: Run 10 samples through the full deployed system ────
# Uses the final fine-tuned models loaded in Step 4 (sent_pipe, risk_pipe, zs_pipe)
# and the helper functions defined in Step 5.

app_eval_rows = []

print('Running 10-sample app evaluation...')
print('='*65)

for tc in APP_TEST_CASES:
    chunks  = chunk_text(tc['transcript'])
    s_score = run_sentiment(chunks, sent_pipe)
    r_score = run_risk(chunks, risk_pipe)
    t_score = run_topic(tc['transcript'], zs_pipe)
    idx     = danger_index(s_score, r_score, t_score)
    signal  = classify_index(idx)

    # Correctness: safe→HEALTHY/CAUTION, cut→DANGER/CRITICAL
    if tc['ground_truth'] == 0:
        correct = signal in ['HEALTHY', 'CAUTION']
    else:
        correct = signal in ['DANGER', 'CRITICAL']

    app_eval_rows.append({
        'Test Sample':   tc['id'],
        'Inputs':        tc['transcript'][:60] + '...',
        'Outputs':       f'{signal} (Index: {idx})',
        'Correct or not': 'Correct' if correct else 'Incorrect',
        # Hidden detail columns for analysis
        'Ground Truth':  'Safe' if tc['ground_truth']==0 else 'Cut/Suspend',
        'Danger Index':  idx,
        'Signal':        signal,
        'Sent Score':    round(s_score, 1),
        'Risk Score':    round(r_score, 1),
        'Topic Score':   round(t_score, 1),
    })

    status = '✅' if correct else '❌'
    print(f"{tc['id']:14s} | GT: {'Safe' if tc['ground_truth']==0 else 'Cut ':4s} "
          f"| {signal:8s} (Index {idx:5.1f}) | {status}")

app_eval_df = pd.DataFrame(app_eval_rows)
correct_n   = (app_eval_df['Correct or not'] == 'Correct').sum()
accuracy_10 = correct_n / len(app_eval_df)

print(f'\nApp Accuracy: {correct_n} out of {len(app_eval_df)} = {accuracy_10:.1%}')


In [ ]:
# ── App Evaluation Step C: Display in template format ────────────────────────

# Table matching professor's exact column format
display_df = app_eval_df[['Test Sample','Inputs','Outputs','Correct or not']].copy()

print('── App Evaluation Results (Template Format) ─────────────────')
display(display_df)

print(f'\nAccuracy: {correct_n} out of {len(app_eval_df)}')
print(f'         = {accuracy_10:.1%}')

# Per-class breakdown
safe_cases = [r for r in app_eval_rows if r['Ground Truth']=='Safe']
cut_cases  = [r for r in app_eval_rows if r['Ground Truth']=='Cut/Suspend']
safe_correct = sum(1 for r in safe_cases if r['Correct or not']=='Correct')
cut_correct  = sum(1 for r in cut_cases  if r['Correct or not']=='Correct')

print(f'\nBreakdown:')
print(f'  Safe transcripts : {safe_correct}/{len(safe_cases)} correct')
print(f'  Cut transcripts  : {cut_correct}/{len(cut_cases)} correct')

# Store for Excel export
app_acc_str = f'{correct_n} out of {len(app_eval_df)}'


In [ ]:
# ── Step 5: Helper functions ─────────────────────────────────────────────────
def chunk_text(text, chunk_size=400, overlap=80):
    words, chunks, step = text.split(), [], chunk_size - overlap
    for i in range(0, len(words), step):
        c = ' '.join(words[i:i+chunk_size])
        if len(c.strip()) > 20:
            chunks.append(c)
    return chunks

def run_sentiment(chunks, pipe):
    negs = []
    for c in chunks:
        r = pipe(c[:512], truncation=True)
        s = {x['label'].lower(): x['score'] for x in r[0]}
        negs.append(s.get('negative', 0))
    avg_neg = float(np.mean(negs))
    return avg_neg * 100   # danger score 0-100

def run_risk(chunks, pipe):
    high_p, med_p = [], []
    for c in chunks:
        r = pipe(c[:512], truncation=True)
        s = {x['label']: x['score'] for x in r[0]}
        high_p.append(s.get('LABEL_2', s.get('High Risk', 0.33)))
        med_p.append( s.get('LABEL_1', s.get('Medium Risk', 0.33)))
    return min(np.mean(high_p)*100 + np.mean(med_p)*50, 100.0)

def run_topic(text, pipe):
    sample = ' '.join(text.split()[:1000])
    r = pipe(sample, DANGER_LABELS, multi_label=True)
    return float(np.mean(r['scores'])) * 100

def danger_index(s, r, t, ws=0.25, wr=0.50, wt=0.25):
    return round(min(max(ws*s + wr*r + wt*t, 0), 100), 1)

def classify_index(idx):
    if idx <= 30:  return 'HEALTHY'
    if idx <= 55:  return 'CAUTION'
    if idx <= 75:  return 'DANGER'
    return 'CRITICAL'

print('Helper functions defined.')

In [ ]:
# ── Step 6: Define test transcripts with known ground-truth labels ───────────
# Ground-truth: 0 = No cut (HEALTHY/CAUTION expected), 1 = Cut/Suspension (DANGER/CRITICAL)
TEST_CASES = [
    {
        'company': 'Sample Corp A',
        'ground_truth': 0,         # dividend maintained
        'expected_signal': 'HEALTHY',
        'transcript': '''Our quarterly results were outstanding. Revenue grew 14% year-over-year
        and free cash flow reached a record 2.4 billion dollars. The payout ratio stands at
        an attractive 38% well within our target range. The board approved a 7% dividend
        increase, reflecting our confidence in continued strong earnings and cash generation.
        Our balance sheet is pristine with net debt at 1.2 times EBITDA and no near-term
        maturities. We are raising full-year guidance and remain committed to growing
        the dividend as a core part of our shareholder return strategy.'''
    },
    {
        'company': 'Sample Corp B',
        'ground_truth': 0,
        'expected_signal': 'CAUTION',
        'transcript': '''This quarter was mixed. While revenue was in line with expectations,
        margin pressure from input cost inflation weighed on earnings. Free cash flow was
        lower than anticipated at 480 million due to elevated working capital. Our payout
        ratio has risen to 68% which is above our target. We remain committed to the dividend
        but acknowledge we need to improve our cash flow conversion. We are taking active
        steps to address margin headwinds and expect improvement in the second half.'''
    },
    {
        'company': 'Sample Corp C (cut risk)',
        'ground_truth': 1,
        'expected_signal': 'DANGER',
        'transcript': '''Results this quarter were significantly below expectations. Revenue
        declined 21% and free cash flow was negative 280 million as we experienced severe
        demand destruction in our core markets. Our leverage ratio has increased to 5.2 times
        which is approaching covenant limits. The board is conducting a comprehensive review
        of our capital allocation including the sustainability of the current dividend at its
        present level. We have drawn 700 million on our revolver and liquidity is now 540
        million. We are exploring all options to strengthen the balance sheet.'''
    },
    {
        'company': 'Sample Corp D (severe)',
        'ground_truth': 1,
        'expected_signal': 'CRITICAL',
        'transcript': '''I must be direct with investors about the severity of our current
        situation. Free cash flow was negative 520 million this quarter; we have fully drawn
        our revolver and our liquidity position stands at 180 million. We have received a
        covenant waiver notice and are in active restructuring discussions with our lenders.
        The board has voted to suspend the quarterly dividend effective immediately to preserve
        capital and stabilize the business. We have engaged a financial restructuring advisor
        and are evaluating all strategic alternatives including potential asset sales and
        equity issuance. Our priority is the survival and long-term viability of the company.'''
    },
    {
        'company': 'Sample Corp E',
        'ground_truth': 0,
        'expected_signal': 'HEALTHY',
        'transcript': '''We delivered another excellent quarter. Adjusted EPS grew 22%
        year-over-year and we generated 1.8 billion in free cash flow. We have now raised
        the dividend for 18 consecutive years. Today we announced an additional 10% dividend
        increase and a new 2 billion share repurchase authorization. Our financial position
        has never been stronger with 4.1 billion in liquidity and a net debt ratio of just
        1.5 times. We are raising the midpoint of our full-year guidance range.'''
    },
    {
        'company': 'Sample Corp F',
        'ground_truth': 1,
        'expected_signal': 'DANGER',
        'transcript': '''Our financial results continued to deteriorate this quarter as
        structural shifts in our industry accelerated. Revenue is down 35% from peak levels
        and we are generating insufficient cash to cover our fixed charges and dividend
        obligations simultaneously. The board reviewed multiple scenarios and determined that
        a 60% reduction in the quarterly dividend is necessary to redirect capital toward
        debt reduction and business investment. This is a difficult but necessary decision
        to ensure our long-term financial stability.'''
    },
]
print(f'Test cases loaded: {len(TEST_CASES)}')

In [ ]:
# ── Step 7: Run full 3-pipeline system on all test cases ─────────────────────
print('Running full system on all test cases ...')
print('=' * 70)

results = []
for tc in TEST_CASES:
    t0     = time.time()
    chunks = chunk_text(tc['transcript'])

    s_score = run_sentiment(chunks, sent_pipe)
    r_score = run_risk(chunks, risk_pipe)
    t_score = run_topic(tc['transcript'], zs_pipe)

    idx     = danger_index(s_score, r_score, t_score)
    signal  = classify_index(idx)
    correct = int(
        (tc['ground_truth'] == 0 and signal in ['HEALTHY','CAUTION']) or
        (tc['ground_truth'] == 1 and signal in ['DANGER','CRITICAL'])
    )
    rt = round(time.time() - t0, 2)

    results.append({
        'Company':       tc['company'],
        'Ground Truth':  'No Cut' if tc['ground_truth']==0 else 'Cut/Suspend',
        'Expected':      tc['expected_signal'],
        'Predicted':     signal,
        'Danger Index':  idx,
        'Sent Score':    round(s_score, 1),
        'Risk Score':    round(r_score, 1),
        'Topic Score':   round(t_score, 1),
        'Correct':       '✅' if correct else '❌',
        'Runtime (s)':   rt,
    })
    print(f"  {tc['company']:25s} | Index: {idx:5.1f} | Signal: {signal:8s} | {'✅' if correct else '❌'}")

results_df = pd.DataFrame(results)
correct_count = results_df['Correct'].str.contains('✅').sum()
print(f'\nSystem Accuracy: {correct_count}/{len(results)} = {correct_count/len(results):.1%}')

In [ ]:
# ── Step 8: Print full results table ────────────────────────────────────────
print('\n── Full Results Table ──────────────────────────────────────────────────')
display(results_df)

In [ ]:
# ── Step 9: Experiment 1 — Base vs Fine-tuned model comparison ───────────────
# Compare FinBERT base vs fine-tuned on sentiment negativity accuracy
print('Experiment 1: Base FinBERT vs Fine-tuned FinBERT (Pipeline 1)')

base_sent_pipe = pipeline('text-classification', model='ProsusAI/finbert',
                           device=-1, top_k=None)

exp1_rows = []
test_sents = [
    ('Free cash flow reached a record high providing excellent dividend coverage.', 'positive'),
    ('The board approved a dividend increase of 8% reflecting strong earnings.', 'positive'),
    ('We expect stable performance and maintain our commitment to the dividend.', 'neutral'),
    ('Revenue was in line with expectations with no material changes to guidance.', 'neutral'),
    ('The payout ratio has risen to unsustainable levels.', 'negative'),
    ('We are reviewing the dividend amid declining free cash flow and rising debt.', 'negative'),
    ('The dividend may need to be reduced to address our liquidity challenges.', 'negative'),
    ('Our financial position deteriorated significantly this quarter.', 'negative'),
]

for sent, true_label in test_sents:
    t0 = time.time()
    base_res = base_sent_pipe(sent[:512])
    base_rt  = round(time.time()-t0, 3)
    base_pred = max(base_res[0], key=lambda x: x['score'])['label'].lower()

    t0 = time.time()
    ft_res = sent_pipe(sent[:512])
    ft_rt  = round(time.time()-t0, 3)
    ft_pred = max(ft_res[0], key=lambda x: x['score'])['label'].lower()

    exp1_rows.append({
        'Sentence':          sent[:55]+'...',
        'True Label':        true_label,
        'Base FinBERT':      base_pred,
        'Fine-tuned FinBERT':ft_pred,
        'Base Correct':      '✅' if base_pred==true_label else '❌',
        'FT Correct':        '✅' if ft_pred==true_label else '❌',
        'Base RT (s)':       base_rt,
        'FT RT (s)':         ft_rt,
    })

exp1_df = pd.DataFrame(exp1_rows)
base_acc = exp1_df['Base Correct'].str.contains('✅').mean()
ft_acc   = exp1_df['FT Correct'].str.contains('✅').mean()
print(f'Base FinBERT accuracy:        {base_acc:.1%}')
print(f'Fine-tuned FinBERT accuracy:  {ft_acc:.1%}')
display(exp1_df)

In [ ]:
# ── Step 10: Experiment 2 — Pipeline ablation study ─────────────────────────
# Test how much each pipeline contributes to correct classification
print('Experiment 2: Pipeline Ablation Study (which pipeline matters most?)')

ablation_rows = []
for tc in TEST_CASES:
    chunks  = chunk_text(tc['transcript'])
    s_score = run_sentiment(chunks, sent_pipe)
    r_score = run_risk(chunks, risk_pipe)
    t_score = run_topic(tc['transcript'], zs_pipe)
    gt_cut  = tc['ground_truth']

    # Full system
    full_idx = danger_index(s_score, r_score, t_score)
    # Only Pipeline 1
    p1_only  = danger_index(s_score, 50, 50)
    # Only Pipeline 2
    p2_only  = danger_index(50, r_score, 50)
    # Only Pipeline 3
    p3_only  = danger_index(50, 50, t_score)
    # No Pipeline 2 (ablate the primary model)
    no_p2    = danger_index(s_score, 50, t_score, ws=0.5, wr=0.0, wt=0.5)

    ablation_rows.append({
        'Company':    tc['company'][:20],
        'GT':         'Cut' if gt_cut else 'Safe',
        'Full System':full_idx,
        'P1 Only':    p1_only,
        'P2 Only':    p2_only,
        'P3 Only':    p3_only,
        'No P2':      no_p2,
    })

abl_df = pd.DataFrame(ablation_rows)
print('\nDanger Index by Pipeline Configuration:')
display(abl_df)

In [ ]:
# ── Step 11: Application performance on Streamlit Cloud ─────────────────────
# Measure end-to-end inference runtime at different transcript lengths
print('Experiment 3: Streamlit App Performance (runtime vs transcript length)')

perf_rows = []
for tc in TEST_CASES:
    chunks = chunk_text(tc['transcript'])
    word_count = len(tc['transcript'].split())

    t0 = time.time()
    s = run_sentiment(chunks, sent_pipe)
    rt_p1 = round(time.time()-t0, 2)

    t0 = time.time()
    r = run_risk(chunks, risk_pipe)
    rt_p2 = round(time.time()-t0, 2)

    t0 = time.time()
    t = run_topic(tc['transcript'], zs_pipe)
    rt_p3 = round(time.time()-t0, 2)

    idx    = danger_index(s, r, t)
    signal = classify_index(idx)

    perf_rows.append({
        'Company':    tc['company'][:20],
        'Words':      word_count,
        'Chunks':     len(chunks),
        'P1 RT (s)':  rt_p1,
        'P2 RT (s)':  rt_p2,
        'P3 RT (s)':  rt_p3,
        'Total RT (s)': round(rt_p1+rt_p2+rt_p3, 2),
        'Index':      idx,
        'Signal':     signal,
    })

perf_df = pd.DataFrame(perf_rows)
print('\nRuntime Results:')
display(perf_df)
print(f'\nAverage total runtime: {perf_df["Total RT (s)"].mean():.2f} s')

In [ ]:
# ── Step 12: Summary accuracy table ─────────────────────────────────────────
print('\n── Experiment Summary ──────────────────────────────────────────────────────')

summary = {
    'Experiment': [
        'Base FinBERT (Pipeline 1, no fine-tuning)',
        'Fine-tuned FinBERT (Pipeline 1)',
        'Base DistilBERT (Pipeline 2, no fine-tuning)',
        'Fine-tuned DistilBERT (Pipeline 2)',
        'BART zero-shot (Pipeline 3)',
        'Full 3-pipeline system',
    ],
    'Accuracy': [
        f'{base_acc:.1%}',
        f'{ft_acc:.1%}',
        'N/A',          # fill in after running base DistilBERT
        'See Notebook 2',
        'N/A (no labels)',
        f'{correct_count/len(results):.1%}',
    ],
    'Avg Runtime (s)': [
        '~0.05',
        '~0.05',
        '~0.04',
        '~0.04',
        '~1.2',
        f'{perf_df["Total RT (s)"].mean():.2f}',
    ],
    'Notes': [
        'Baseline, no task-specific fine-tuning',
        'Fine-tuned on Financial PhraseBank',
        'Baseline classifier',
        'PRIMARY model — fine-tuned on dividend risk dataset',
        'Zero-shot; no fine-tuning needed',
        'All 3 pipelines fused via weighted index',
    ],
}
summary_df = pd.DataFrame(summary)
display(summary_df)

In [ ]:
# ── Step 13: Export all results to Excel ─────────────────────────────────────
output_file = 'Experimental_Results.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    p1_selection_df.to_excel(writer, sheet_name='P1 Model Selection',  index=False)
    p2_selection_df.to_excel(writer, sheet_name='P2 Model Selection',  index=False)
    display_df.to_excel(     writer, sheet_name='App 10-Sample Eval',  index=False)
    summary_df.to_excel(writer, sheet_name='Summary',           index=False)
    results_df.to_excel(writer, sheet_name='Full System Test',  index=False)
    exp1_df.to_excel(   writer, sheet_name='Exp1 Sentiment',    index=False)
    abl_df.to_excel(    writer, sheet_name='Exp2 Ablation',     index=False)
    perf_df.to_excel(   writer, sheet_name='Exp3 Performance',  index=False)

print(f'✅ Saved: {output_file}')
print('   Submit this file as your Experimental_Results.xlsx submission.')